# Conditional logistic regression — dataset construction

Builds a **risk-set matched case-control** dataset for conditional logistic regression (CLR).

## Why this design

The full panel has 0.02% positives, which is why Precision@K sat at zero. Matched case-control
fixes that by *sampling*, not by changing the question:

- A **case** is a firm at the moment it took its **first Lloyds charge** (its `index_date`).
- Its **controls** are firms that were *also at risk* on that same date — they existed, and had no
  Lloyds charge yet — but did not convert then.
- A **stratum** = one case + its `N_CONTROLS` controls, all measured on the **same date**.

Because everyone in a stratum shares the date, calendar effects (interest rates, seasonality,
market conditions) are **matched out by design** — CLR conditions them away rather than modelling
them. And prevalence rises from 0.02% to `1/(1+N_CONTROLS)`, which is a tractable problem.

Every feature is computed **as of that stratum's `index_date`**, so nothing from the future leaks in.

## What you control

Edit the config cell below — `STUDY_START` / `STUDY_END` set the cut-off window, `N_CONTROLS` sets
the matching ratio. Everything downstream follows.

Uses only `pandas` and `numpy` for construction (plus `scipy.optimize` for the optional fit at the end).

## 1 · Config — the knobs you manage

In [1]:
# --- portable paths: resolve the project root from ANY working directory ---
import sys
from pathlib import Path
_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "API").is_dir())
sys.path.insert(0, str(_ROOT))
from paths import COMPANIES_CSV, CHARGES_CSV, API_DIR

import numpy as np
import pandas as pd

# ============================ SETTINGS ============================
# Cut-off window: a firm is a CASE if its FIRST Lloyds charge falls inside this
# window. Widen it for more cases; narrow it to focus on a recent regime.
STUDY_START = pd.Timestamp("2023-01-01")
STUDY_END   = pd.Timestamp("2025-12-31")

N_CONTROLS  = 50        # controls sampled per case (5 -> ~17% positives per stratum)
SEED        = 42       # reproducible control sampling
MIN_AGE_DAYS = 0       # optional: require controls to be at least this old at index_date

OUT_CSV = API_DIR / "flat_conditional.csv"
# ==================================================================

rng = np.random.default_rng(SEED)
print(f"window      : {STUDY_START.date()}  ->  {STUDY_END.date()}")
print(f"matching    : 1 case : {N_CONTROLS} controls")
print(f"output      : {OUT_CSV}")

window      : 2023-01-01  ->  2025-12-31
matching    : 1 case : 50 controls
output      : /Users/natchalin_/Projects/final_project/Lloyds/API/flat_conditional.csv


## 2 · Load the companies and their charge history

`charges_history.csv` is the label source. If it looks short, run **Stage 4b** in the CH pipeline
first — it rebuilds the file from the JSONs already on disk (no API calls).

In [2]:
# ONE company table now; is_sme flags the modelling population. Non-SME rows are
# kept in the file as the "already checked, do not re-pull" cache, so filter here.
companies = pd.read_csv(COMPANIES_CSV, dtype=str, low_memory=False)
companies = companies[companies["is_sme"].astype(str).str.lower().eq("true")].copy()
charges   = pd.read_csv(CHARGES_CSV, dtype=str)

companies["born"] = pd.to_datetime(companies["date_of_creation"], errors="coerce")
companies = companies.dropna(subset=["born"]).reset_index(drop=True)

charges["created_on"] = pd.to_datetime(charges["created_on"], errors="coerce")
charges["is_lloyds"]  = charges["is_lloyds"].astype(str).str.lower().eq("true")
charges = charges.dropna(subset=["created_on"])

# first Lloyds charge per firm = the conversion event we are modelling
first_lloyds = charges[charges["is_lloyds"]].groupby("com_num")["created_on"].min()
nonlloyds    = charges[~charges["is_lloyds"]]          # safe as features (Lloyds = label)

companies["first_lloyds"] = companies["com_num"].map(first_lloyds)

print(f"companies          : {len(companies):,}")
print(f"charge rows        : {len(charges):,}")
print(f"ever-Lloyds firms  : {companies['first_lloyds'].notna().sum():,}")

companies          : 567,835
charge rows        : 399,700
ever-Lloyds firms  : 19,630


## 3 · Define the cases

A case is a firm whose **first** Lloyds charge lands inside the study window. The date of that
charge becomes the stratum's `index_date` — the moment we ask "why this firm, now?"

In [3]:
is_case = (companies["first_lloyds"] >= STUDY_START) & (companies["first_lloyds"] <= STUDY_END)
cases = companies[is_case].copy()
cases["index_date"] = cases["first_lloyds"]

print(f"cases in window: {len(cases):,}")
print("\ncases per year:")
print(cases["index_date"].dt.year.value_counts().sort_index().to_string())

cases in window: 1,181

cases per year:
index_date
2023    344
2024    380
2025    457


## 4 · Sample the controls (risk-set / incidence-density sampling)

For each case we look at who was **at risk** on that exact date — alive, and not yet a Lloyds
customer — and draw `N_CONTROLS` of them at random.

Two things that look odd but are correct and required for this design to be unbiased:

- a control may **later** become a case in its own stratum, and
- a firm may be drawn as a control in **several** strata.

Both are standard in incidence-density sampling.

In [4]:
# numpy views so the per-case masks stay fast
born_i  = companies["born"].values.astype("datetime64[D]").astype(int)
lloyd_i = companies["first_lloyds"].values.astype("datetime64[D]").astype(float)  # NaN = never
all_pos = np.arange(len(companies))
case_pos = set(cases.index)

strata = []                 # (stratum_id, company row position, is_case, index_date)
skipped = 0

for sid, (case_idx, row) in enumerate(cases.iterrows()):
    t = np.datetime64(row["index_date"], "D").astype(int)

    # AT RISK at t: already existed, and no Lloyds charge on or before t
    at_risk = (born_i <= t - MIN_AGE_DAYS) & (np.isnan(lloyd_i) | (lloyd_i > t))
    at_risk[case_idx] = False                      # the case is added separately

    pool = all_pos[at_risk]
    if len(pool) < N_CONTROLS:
        skipped += 1
        continue

    picks = rng.choice(pool, size=N_CONTROLS, replace=False)
    strata.append((sid, case_idx, 1, row["index_date"]))
    for p in picks:
        strata.append((sid, p, 0, row["index_date"]))

matched = pd.DataFrame(strata, columns=["stratum", "row", "is_case", "index_date"])
print(f"strata built : {matched['stratum'].nunique():,}   (skipped {skipped} with a thin risk set)")
print(f"rows         : {len(matched):,}  =  {matched['is_case'].sum():,} cases "
      f"+ {(~matched['is_case'].astype(bool)).sum():,} controls")

strata built : 1,181   (skipped 0 with a thin risk set)
rows         : 60,231  =  1,181 cases + 59,050 controls


## 5 · Point-in-time features, measured at each stratum's `index_date`

Every feature below uses **only** information available on that date. The non-Lloyds charge
features are cut at `index_date` — this is exactly the bug that produced negative
`yrs_since_nonlloyds_chg` values in `flat.csv`, so it is enforced explicitly here.

`sector`, `region` and `account_type` are still **current snapshots** (Companies House does not
serve history for them cheaply) — they are near-static, but note it as a caveat.

In [5]:
# attach the company columns to each matched row
info = companies.reset_index(drop=True)

# `region` comes from Part 5 of GDELT.ipynb, but the CH pipeline's Stage 2 rebuild
# drops it (it is not in ENRICH_COLS), so join defensively and carry a placeholder.
_cols = ["com_num", "name", "born", "sic_code", "account_type", "accounts_overdue"]
HAS_REGION = "region" in info.columns and info["region"].notna().mean() > 0.5
if HAS_REGION:
    _cols.insert(4, "region")
else:
    print("NOTE: 'region' missing from the SME file -> using 'unknown'. "
          "Re-run Part 5 of GDELT.ipynb to repopulate it.")

m = matched.join(info[_cols], on="row")
if not HAS_REGION:
    m["region"] = "unknown"
m["region"] = m["region"].fillna("unknown")     # firms with no post_code

# ---- age at index_date ----------------------------------------------------------
m["age_years"] = (m["index_date"] - m["born"]).dt.days / 365.25

# ---- non-Lloyds charge history STRICTLY BEFORE index_date ------------------------
# Cross firm x stratum against the charge list, then cut at each stratum's own date.
nl = nonlloyds[["com_num", "created_on", "persons_entitled"]]
j = m[["stratum", "com_num", "index_date"]].merge(nl, on="com_num", how="left")
j = j[j["created_on"] <= j["index_date"]]                # <-- the point-in-time cut

agg = j.groupby(["stratum", "com_num"]).agg(
    prior_charges=("created_on", "size"),
    last_charge=("created_on", "max"),
    n_lenders=("persons_entitled", "nunique"),
).reset_index()

m = m.merge(agg, on=["stratum", "com_num"], how="left")
m["prior_charges"] = m["prior_charges"].fillna(0).astype(int)
m["n_lenders"]     = m["n_lenders"].fillna(0).astype(int)
m["has_prior_charge"] = (m["prior_charges"] > 0).astype(int)

# years since the most recent prior charge; 0 + flag when there is none
m["yrs_since_charge"] = ((m["index_date"] - m["last_charge"]).dt.days / 365.25).fillna(0.0)

# ---- tidy categoricals ----------------------------------------------------------
SECTIONS = [(1,3,"A: Agriculture"),(5,9,"B: Mining"),(10,33,"C: Manufacturing"),(35,35,"D: Utilities"),
            (36,39,"E: Water/Waste"),(41,43,"F: Construction"),(45,47,"G: Retail/Wholesale"),
            (49,53,"H: Transport"),(55,56,"I: Accommodation/Food"),(58,63,"J: Information/Comms"),
            (64,66,"K: Finance/Insurance"),(68,68,"L: Real Estate"),(69,75,"M: Professional/Scientific"),
            (77,82,"N: Admin Support"),(84,84,"O: Public Admin"),(85,85,"P: Education"),
            (86,88,"Q: Health/Social"),(90,93,"R: Arts/Recreation"),(94,96,"S: Other Services"),
            (97,98,"T: Household Activities"),(99,99,"U: Extraterritorial")]

def to_section(code):
    try:
        div = int(str(code).strip()[:2])
    except (ValueError, TypeError):
        return None
    if div == 98:
        return "L: Real Estate"                     # resident property mgmt -> Real Estate
    for lo, hi, name in SECTIONS:
        if lo <= div <= hi:
            return name
    return None

m["sector"] = m["sic_code"].map(to_section)
m["accounts_overdue"] = m["accounts_overdue"].map(
    {"True": 1, True: 1, "False": 0, False: 0}).fillna(0).astype(int)

print("features built. sanity checks:")
print(f"  negative yrs_since_charge : {(m['yrs_since_charge'] < 0).sum()}   (must be 0)")
print(f"  negative age_years        : {(m['age_years'] < 0).sum()}   (must be 0)")
print(f"  sector missing            : {m['sector'].isna().sum():,}")

features built. sanity checks:
  negative yrs_since_charge : 0   (must be 0)
  negative age_years        : 0   (must be 0)
  sector missing            : 0


## 6 · Save the matched dataset

In [6]:
KEEP = ["stratum", "index_date", "com_num", "name", "is_case",
        "age_years", "prior_charges", "has_prior_charge", "yrs_since_charge",
        "n_lenders", "sector", "region", "account_type", "accounts_overdue"]

clr = m[KEEP].sort_values(["stratum", "is_case"], ascending=[True, False]).reset_index(drop=True)

# a stratum is only usable if it kept exactly one case and its full set of controls
sizes = clr.groupby("stratum")["is_case"].agg(["size", "sum"])
good  = sizes[(sizes["size"] == N_CONTROLS + 1) & (sizes["sum"] == 1)].index
clr   = clr[clr["stratum"].isin(good)].reset_index(drop=True)

clr.to_csv(OUT_CSV, index=False)
print(f"saved {len(clr):,} rows / {clr['stratum'].nunique():,} strata  ->  {OUT_CSV}")
print(f"positives: {clr['is_case'].mean():.1%} of rows\n")
clr.head(N_CONTROLS + 1)

saved 60,231 rows / 1,181 strata  ->  /Users/natchalin_/Projects/final_project/Lloyds/API/flat_conditional.csv
positives: 2.0% of rows



,stratum,index_date,com_num,name,is_case,age_years,prior_charges,has_prior_charge,yrs_since_charge,n_lenders,sector,region,account_type,accounts_overdue
0,0,2025-12-01,15458172,37TD LIMITED,1,1.834360,1,1,1.694730,1,L: Real Estate,London,micro-entity,0
1,0,2025-12-01,13784715,AV ESTATES LIMITED,0,3.986311,1,1,2.173854,1,L: Real Estate,London,micro-entity,0
2,0,2025-12-01,08250085,BRIGHT CHILD (HORNSEY) LTD,0,13.138946,0,0,0.000000,0,P: Education,South East,total-exemption-full,0
3,0,2025-12-01,08904149,BOGUE PROPERTIES LIMITED,0,11.778234,1,1,5.015743,1,L: Real Estate,West Midlands,total-exemption-full,0
4,0,2025-12-01,13614060,ABLEY LETTINGS LIMITED,0,4.224504,2,1,1.590691,1,L: Real Estate,North East,total-exemption-full,0
5,0,2025-12-01,12867330,KEITH RHODES HOLDINGS LIMITED,0,5.226557,0,0,0.000000,0,L: Real Estate,South West,total-exemption-full,0
6,0,2025-12-01,16374433,RECUR PROPERTY 001 LIMITED,0,0.648871,0,0,0.000000,0,L: Real Estate,London,total-exemption-full,0
7,0,2025-12-01,15566915,ESONGRACE STAYS LTD,0,1.711157,0,0,0.000000,0,L: Real Estate,London,micro-entity,0
8,0,2025-12-01,13984917,HARLSEY PROPERTY LIMITED,0,3.709788,2,1,1.733060,2,L: Real Estate,North East,micro-entity,0
9,0,2025-12-01,15768671,AP INVEST LTD,0,1.475702,1,1,0.774812,1,L: Real Estate,East Midlands,total-exemption-full,0


## 7 · Sanity checks

Cases and controls should differ on the things that matter and be identical on the thing we
matched (the date). If a feature looks *identical* across cases and controls it carries no
information; if it looks wildly different, check it is not leakage.

In [7]:
num = ["age_years", "prior_charges", "has_prior_charge", "yrs_since_charge", "n_lenders"]
print("mean by group (1 = case, 0 = control):")
print(clr.groupby("is_case")[num].mean().round(3).to_string())

print("\ntop sectors, case share within stratum:")
tab = clr.groupby("sector")["is_case"].agg(["size", "mean"]).sort_values("size", ascending=False)
tab.columns = ["rows", "case_rate"]
print(tab.head(10).round(3).to_string())

print(f"\nstrata all same size? {clr.groupby('stratum').size().nunique() == 1}")
print(f"exactly one case per stratum? {(clr.groupby('stratum')['is_case'].sum() == 1).all()}")

mean by group (1 = case, 0 = control):
         age_years  prior_charges  has_prior_charge  yrs_since_charge  n_lenders
is_case                                                                         
0            7.054          1.312             0.457             1.759      0.796
1            7.454          1.023             0.329             1.698      0.560

top sectors, case share within stratum:
                             rows  case_rate
sector                                      
L: Real Estate              26930      0.011
G: Retail/Wholesale          5720      0.025
Q: Health/Social             5216      0.026
F: Construction              3578      0.030
K: Finance/Insurance         2866      0.029
M: Professional/Scientific   2806      0.026
N: Admin Support             2780      0.029
C: Manufacturing             2519      0.025
I: Accommodation/Food        1884      0.023
J: Information/Comms         1193      0.017

strata all same size? True
exactly one case per stratum

## 8 · (Optional) fit the conditional logistic regression

The conditional likelihood is simpler than it sounds. Within a stratum we ask: **given that exactly
one of these firms converted, which one was it?** That is a softmax over the stratum:

$$P(\text{case is firm } i) = \frac{e^{x_i\beta}}{\sum_j e^{x_j\beta}}$$

So fitting is just maximising the log-probability of picking the true case in every stratum — about
ten lines with `scipy.optimize`. No stratum intercepts appear anywhere: they cancel in the ratio,
which is precisely what "conditional" means here.

Note this is the same *ranking* idea as Precision@K — the model is rewarded for scoring the real
converter above its matched peers.

In [8]:
import math
import numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp

# --- design matrix: numeric features + two collapsed dummies (stay parsimonious) ---
X = clr[["age_years", "prior_charges", "has_prior_charge",
         "yrs_since_charge", "n_lenders", "accounts_overdue"]].astype(float).copy()

HEAVY = {"C: Manufacturing", "F: Construction", "L: Real Estate",
         "H: Transport", "B: Mining", "D: Utilities"}
X["sector_heavy"] = clr["sector"].isin(HEAVY).astype(float)
X["london_se"]    = clr["region"].isin(["London", "South East"]).astype(float)
if X["london_se"].nunique() == 1:          # region unavailable -> the dummy is constant
    X = X.drop(columns="london_se")        # drop it rather than divide by zero SD

names = list(X.columns)
Xz = ((X - X.mean()) / X.std().replace(0, 1)).values      # standardise -> coefs are per 1 SD

K = N_CONTROLS + 1
S = Xz.reshape(len(clr) // K, K, Xz.shape[1])             # (strata, K, features); row 0 = the case


def neg_log_lik(beta):
    """-log P(the true case is the one that converted), summed over strata."""
    scores = S @ beta                                     # (strata, K)
    return -(scores[:, 0] - logsumexp(scores, axis=1)).sum()


def gradient(beta):
    """d/dbeta of the above: (features of the case) - (stratum-average features)."""
    scores = S @ beta
    prob = np.exp(scores - logsumexp(scores, axis=1, keepdims=True))   # (strata, K)
    expected = (prob[:, :, None] * S).sum(axis=1)                      # (strata, features)
    return -(S[:, 0, :] - expected).sum(axis=0)


# passing the analytic gradient is what makes this converge cleanly; without it
# scipy's finite-difference approximation stalls just short of the optimum.
fit = minimize(neg_log_lik, np.zeros(len(names)), method="BFGS", jac=gradient)

se = np.sqrt(np.diag(fit.hess_inv))
out = pd.DataFrame({"coef": fit.x, "se": se,
                    "odds_ratio": np.exp(fit.x), "z": fit.x / se}, index=names)
out["p"] = [2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2)))) for z in out["z"]]

print(f"converged: {fit.success}  |  strata: {S.shape[0]:,}  |  "
      f"max|gradient|: {np.abs(gradient(fit.x)).max():.2e}\n")
print(out.round(3).to_string())
print("\nodds_ratio > 1 => raises conversion odds (per 1 SD). |z| > 1.96 => significant at 5%.")

converged: True  |  strata: 1,181  |  max|gradient|: 2.06e-06

                   coef     se  odds_ratio      z      p
age_years        -0.009  0.037       0.991 -0.246  0.805
prior_charges     0.072  0.033       1.074  2.135  0.033
has_prior_charge -0.291  0.056       0.748 -5.214  0.000
yrs_since_charge  0.114  0.037       1.120  3.066  0.002
n_lenders        -0.035  0.063       0.966 -0.554  0.580
accounts_overdue  0.068  0.019       1.071  3.673  0.000
sector_heavy     -0.199  0.032       0.820 -6.210  0.000
london_se        -0.020  0.031       0.980 -0.667  0.505

odds_ratio > 1 => raises conversion odds (per 1 SD). |z| > 1.96 => significant at 5%.
